# Module 3.1: Answer Hotel Questions and Record Booking Requests

Use a fixed hotel search and a protected reservation command with your configured **Neo4j database** and **Amazon Bedrock**. This notebook creates no AWS resources.

This notebook retrieves hotel facts, responds when the graph has no answer, enforces Neo4j's maximum-guests rule, and safely retries a reservation request.

**Overview**

- **Grounded answer:** The agent answers using only hotel facts returned from Neo4j.
- **Hybrid search:** One search combines vector and full-text matching.
- **Idempotent request:** A request can run again without creating a duplicate reservation.

## Neo4j and AWS roles

| Neo4j owns | AWS owns |
|---|---|
| Connected hotel knowledge: hotels, amenities, ratings, policies | Amazon Bedrock reasons over the retrieved context |
| The vector index `hotel_chunk_embeddings` and full-text index `hotel_chunk_fulltext` | Amazon Nova 2 creates the query embedding |
| The reviewed Cypher traversal that enriches a matched `Chunk` with its hotel |  |
| The maximum-guests rule and the idempotent `ReservationRequest` write |  |

The notebook uses one fixed `HybridCypherRetriever`. It uses `NAIVE` fusion, `top_k=5`, and one reviewed traversal. It accepts one `query` argument. Module 2 compares retrieval roles and selects this fixed Hybrid-Cypher pattern for the application.

In [ ]:
import os
import sys
from pathlib import Path


def locate_notebooks_root():
    override = os.environ.get("WORKSHOP_NOTEBOOKS_DIR")
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "workshop").is_dir():
            return candidate
        raise RuntimeError(
            "WORKSHOP_NOTEBOOKS_DIR must contain the workshop package"
        )

    start = Path.cwd().resolve()
    for candidate in (start, start / "notebooks", start.parent):
        if (candidate / "workshop").is_dir():
            return candidate
    raise RuntimeError(
        "Run from the repository root, notebooks/, or this module "
        "directory; or set WORKSHOP_NOTEBOOKS_DIR."
    )


NOTEBOOKS_ROOT = locate_notebooks_root()
REPO_ROOT = NOTEBOOKS_ROOT.parent
MODULE_DIR = NOTEBOOKS_ROOT / "03-grounded-booking-agent"
for path in (NOTEBOOKS_ROOT, MODULE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"Workshop root: {REPO_ROOT}")


In [ ]:
import json
import os
import uuid
from datetime import date, timedelta

import boto3
from dotenv import load_dotenv

from workshop.aws_region import configure_aws_region
from workshop.bedrock_providers import default_model_id
from workshop.contracts import (
    MAX_GUESTS,
    OVER_LIMIT_GUESTS,
    ReservationReason,
    ReservationStatus,
)
from workshop.fixtures import (
    HERO_NAME,
    HERO_SOURCE,
    apply_reservation_fixtures,
    load_manifest,
    readiness_problems,
)
from workshop.hybrid_retrieval import (
    GROUNDING_INSTRUCTIONS,
    Neo4jConfig,
    search_hotel_knowledge,
)
from reservation_command import create_reservation_request
from neo4j import GraphDatabase

load_dotenv(NOTEBOOKS_ROOT / ".env")
load_dotenv(REPO_ROOT / ".env")
load_dotenv(REPO_ROOT / "CONFIG.txt")

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = NEO4J_READY and BEDROCK_READY

AWS_REGION = configure_aws_region()
MODEL_ID = default_model_id()
HERO_QUESTION = f"What amenities and guest rating does {HERO_NAME} have?"
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

if not NEO4J_READY:
    print("Neo4j is not configured, so live cells will be skipped. Set NEO4J_URI/USERNAME/PASSWORD/DATABASE.")
if not BEDROCK_READY:
    print("AWS credentials are not configured, so live cells will be skipped.")
if RETRIEVAL_READY:
    print("Participant retrieval is configured. Ready to retrieve the fixture hotel.")

## 1. Prepare the graph for hotel search and booking

Run the cell below to set up the Module 3 graph data. It applies fixture hotel IDs, uniqueness constraints, and the maximum-guests rule. You can run it more than once. It also checks that both retrieval indexes are online and the example hotel is present. It does not change the canonical hotel facts.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph preparation: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        driver.verify_connectivity()
        problems = apply_reservation_fixtures(driver, config.database, manifest)
        if not problems:
            problems = readiness_problems(driver, config.database, manifest)
    finally:
        driver.close()
    if problems:
        print("Graph is not ready:")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("Participant graph is ready: both indexes online, fixtures applied, rule present.")

## 2. Search for one hotel's amenities and rating

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

Run this search to find the hotel's amenities and guest rating. Full-text search matches the hotel name. Vector search matches the request for amenities and a rating. The reviewed traversal returns the connected hotel, its amenities, its rating, and the stable `hotel_id`. The results are the highest-scoring grounded matches.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping hotel details question: retrieval is not configured.")
else:
    results = search_hotel_knowledge(HERO_QUESTION)
    top = results[0]
    print(f"Question: {HERO_QUESTION}\n")
    print(f"Hotel: {top['hotel_name']} | hotel_id={top['hotel_id']}")
    print(f"Combined hybrid score: {top['combined_score']:.4f}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Exact matched terms: {', '.join(top['exact_terms']) or 'none'}")
    print(f"Amenities: {', '.join(top['amenities'])}")
    print(f"Amenity count: {len(top['amenities'])}")
    print("\nChunk text:")
    print(top["chunk_text"][:600])

### How the search finds hotel facts

- **Vector matching:** It finds text with the same meaning as "amenities and guest rating".
- **Full-text matching:** It finds the exact hotel name and location terms.
- **Reviewed Cypher traversal:** It follows the matched `Chunk` to its hotel. It returns up to 12 connected amenities, the guest rating, and the opaque `hotel_id`. The reservation command uses this `hotel_id` to identify the hotel.

This notebook fixes the fusion behavior and `top_k`. The retrieval contract stays exact across runs. Module 2 compares the retrieval roles and chooses this configuration for the application. The graph fields come from the data extracted into Neo4j. The returned source `Chunk` and provenance show where those fields came from. Model wording can vary between runs. The retrieval contract stays the same each time.

## 3. Handle a question that the graph cannot answer

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph stores hotel knowledge. It does not store live inventory. The agent must **abstain** when the retrieved context cannot confirm availability. The next cells give a small local agent the same search tool and the workshop's grounding instructions. The agent answers the hotel-details question first. It then answers the availability question.

### Strands agent basics

This section introduces the Strands Agents SDK. The next cell builds a local agent with four parts.

**Brief overview**

- **`Agent`:** It sends questions to the model and runs the tools it requests.
- **`BedrockModel`:** It connects the agent to the Amazon Bedrock model named by the model ID.
- **`@tool`:** It marks a Python function as a tool the model can call.
- **System prompt:** It defines which facts the model may use and when it must say the graph has no answer.

The next cell turns `search_hotel_knowledge` into a tool. The agent calls this tool for every new hotel question. It answers from the returned facts. It says it cannot determine the answer when the graph lacks the facts the question needs.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping grounded agent: retrieval is not configured.")
else:
    from strands import Agent, tool
    from strands.models import BedrockModel

    @tool
    def search_hotel_knowledge_tool(query: str) -> str:
        """Search grounded hotel context and return bounded JSON facts."""
        return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

    class GroundedBedrockModel(BedrockModel):
        """Requires a tool call before the model may answer a fresh question.

        `tool_choice` is not a real `Agent(...)`/`BedrockModel(...)` construction
        argument: Strands only threads `tool_choice` through `Model.stream()` for
        its own internal structured-output calls, so passing it at construction
        time is silently accepted into an unused config key and never reaches a
        request. Overriding `stream()` -- the public method every model
        provider implements -- is the supported way to force tool use for a
        specific call.

        Forces `tool_choice={"any": {}}` only when the latest message is a
        fresh question (the model has not yet returned a tool result for it),
        so grounding is enforced by the API instead of by system-prompt wording
        alone. Once a tool result comes back, tool choice reverts to the
        model's normal "auto" behavior so the final answer can be free text.
        """

        async def stream(
            self,
            messages,
            tool_specs=None,
            system_prompt=None,
            *,
            tool_choice=None,
            **kwargs,
        ):
            last_message = messages[-1] if messages else None
            fresh_question = bool(
                tool_specs
                and last_message
                and last_message.get("role") == "user"
                and not any(
                    "toolResult" in block for block in last_message.get("content", [])
                )
            )
            if fresh_question and tool_choice is None:
                tool_choice = {"any": {}}
            async for event in super().stream(
                messages, tool_specs, system_prompt, tool_choice=tool_choice, **kwargs
            ):
                yield event

    grounded_agent = Agent(
        model=GroundedBedrockModel(
            model_id=MODEL_ID, region_name=AWS_REGION
        ),
        tools=[search_hotel_knowledge_tool],
        system_prompt=(
            "You are a grounded hotel-information assistant. Call "
            "search_hotel_knowledge_tool before answering any hotel question.\n\n"
            + GROUNDING_INSTRUCTIONS
        ),
    )
    print(grounded_agent(HERO_QUESTION))

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping availability question: retrieval is not configured.")
else:
    availability_response = grounded_agent(AVAILABILITY_QUESTION)
    print(availability_response)

    availability_text = str(availability_response).lower()
    fabricated_availability_claims = (
        "yes, rooms are available",
        "is available next weekend",
        "availability is guaranteed",
        "we guarantee availability",
        "yes, it guarantees",
        "confirmed availability",
        "rooms are open",
    )
    assert not any(
        claim in availability_text for claim in fabricated_availability_claims
    ), availability_text
    abstention_markers = (
        "cannot determine",
        "cannot be determined",
        "cannot confirm",
        "cannot be confirmed",
        "can't confirm",
        "unable to confirm",
        "no evidence",
        "does not guarantee",
        "doesn't guarantee",
    )
    assert any(marker in availability_text for marker in abstention_markers), availability_text

## 4. Reject an over-limit reservation request

The Neo4j maximum-guests rule sets a 10-guest limit. The command reads and enforces this rule inside the write transaction. It rejects a request for 15 guests before it creates a node.

These cells require your Aura connection through `NEO4J_READY`. They do not use Bedrock. The local fixture manifest supplies the example `hotel_id`, so this write example does not need live search results. The notebook calculates dates from the current day to keep the example valid.

In [ ]:
if not NEO4J_READY:
    print("Skipping rule rejection: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    hero_id = manifest.hotels[HERO_SOURCE]
    check_in = (date.today() + timedelta(days=30)).isoformat()
    check_out = (date.today() + timedelta(days=32)).isoformat()
    REQUEST_ID = str(uuid.uuid4())
    print(f"Hero hotel_id from fixture manifest: {hero_id}")
    print(f"Caller-created request_id for retries: {REQUEST_ID}")
    print(f"Stay: {check_in} to {check_out}\n")

    over_limit_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": OVER_LIMIT_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        rejected = create_reservation_request(
            over_limit_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print(json.dumps(rejected, indent=2))

    assert rejected["status"] == ReservationStatus.REJECTED.value, rejected
    assert rejected["reason_code"] == ReservationReason.MAX_GUESTS_EXCEEDED.value, rejected
    assert rejected["hotel_id"] == hero_id, rejected
    assert rejected["max_guests"] == MAX_GUESTS, rejected

## 5. Create one reservation and safely retry it

Submit a request within the 10-guest limit, then submit it again with the same `request_id`. The first request creates one `ReservationRequest` and links it to the hotel with a `FOR_HOTEL` relationship. The second request finds the existing request and returns `duplicate=true` with its original `created_at`. The uniqueness constraint prevents a second node with the same `request_id`.

In [ ]:
if not NEO4J_READY:
    print("Skipping valid write: Neo4j is not configured.")
else:
    valid_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": MAX_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        accepted = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
        replay = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print("First delivery:")
    print(json.dumps(accepted, indent=2))
    print("\nSame request_id re-delivered:")
    print(json.dumps(replay, indent=2))

    assert accepted["status"] == ReservationStatus.ACCEPTED.value, accepted
    assert accepted["hotel_id"] == hero_id, accepted
    assert accepted["duplicate"] is False, accepted

    assert replay["status"] == ReservationStatus.ACCEPTED.value, replay
    assert replay["hotel_id"] == hero_id, replay
    assert replay["duplicate"] is True, replay

## 6. Verify the reservation in Neo4j

Use the stable `request_id` to confirm that the graph has one accepted request linked to one hotel.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        with driver.session(database=config.database) as session:
            for record in session.run(query, rid=REQUEST_ID):
                print(dict(record))
    finally:
        driver.close()

## Continue to the next modules

- **Module 4:** It deploys the retrieval functions through Amazon Bedrock AgentCore Gateway and AWS Lambda.
- **Module 5:** It packages a deployment-oriented version of the agent for AgentCore Runtime with Docker, Secrets Manager, and IAM boundaries.
- **Module 6:** It adds actor-scoped, cross-session graph memory with provenance and direct correction.

All steps above run against your own Aura instance and Amazon Bedrock. They create no AWS resources.